In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

df= pd.read_csv("homework_1.1.csv")

In [7]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1000 non-null   float64
 1   X2      1000 non-null   float64
 2   X3      1000 non-null   float64
 3   Y       1000 non-null   float64
dtypes: float64(4)
memory usage: 31.4 KB


In [18]:
X = df[['X1', 'X2', 'X3']]
Y = df['Y']

X = sm.add_constant(X)

model = sm.OLS(Y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                        19:32:59   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0026      0.016      0.166      0.8

In [15]:

Y = df['Y']

variables = ['X1', 'X2', 'X3']

results = []

for var in variables:
    X_simple = sm.add_constant(df[[var]])
    simple_model = sm.OLS(Y, X_simple).fit()
    simple_coef = simple_model.params[var]

    results.append({
        'Variable': var,
        'Simple Regression Coef': simple_coef
    })

X_multiple = sm.add_constant(df[variables])
multiple_model = sm.OLS(Y, X_multiple).fit()
for row in results:
    var = row['Variable']
    multiple_coef = multiple_model.params[var]
    row['Multiple Regression Coef'] = multiple_coef
    row['Absolute Difference'] = abs(row['Simple Regression Coef'] - multiple_coef)

comparison = pd.DataFrame(results)
comparison = comparison.sort_values(by='Absolute Difference', ascending=False)

print(comparison)


  Variable  Simple Regression Coef  Multiple Regression Coef  \
1       X2                4.083613                  1.964569   
0       X1                1.841761                  1.007138   
2       X3                3.097041                  2.975489   

   Absolute Difference  
1             2.119044  
0             0.834623  
2             0.121553  


In [23]:
from sklearn.neighbors import NearestNeighbors
df2= pd.read_csv("homework_1.2.csv")
df2.head()

,X,Y,Z
0,0,0.548814,0.548814
1,1,1.215189,0.715189
2,0,0.602763,0.602763
3,0,0.544883,0.544883
4,0,0.423655,0.423655


In [25]:
treated = df2[df2['X'] == 1]
control = df2[df2['X'] == 0]
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['Z']])

NearestNeighbors(n_neighbors=1)

In [30]:
distances, indices = nn.kneighbors(treated[['Z']])
matched_controls = control.iloc[indices.flatten()]

In [34]:

matched_data = treated.reset_index(drop=True).copy()

matched_data['Matched_Control_Z'] = matched_controls['Z'].values
matched_data['Distance'] = distances.flatten()

print(matched_data.sort_values(by='Distance', ascending=False))

    X         Y         Z  Matched_Control_Z  Distance
29  1  1.488374  0.988374           0.778157  0.210217
10  1  1.478618  0.978618           0.778157  0.200462
37  1  1.476761  0.976761           0.778157  0.198604
36  1  1.476459  0.976459           0.778157  0.198303
4   1  1.463663  0.963663           0.778157  0.185506
15  1  1.444669  0.944669           0.778157  0.166512
20  1  1.443748  0.943748           0.778157  0.165591
43  1  1.429296  0.929296           0.778157  0.151139
7   1  1.425597  0.925597           0.778157  0.147440
3   1  1.391773  0.891773           0.778157  0.113616
9   1  1.370012  0.870012           0.778157  0.091855
35  1  1.337945  0.837945           0.778157  0.059788
8   1  1.332620  0.832620           0.778157  0.054463
47  1  1.328940  0.828940           0.778157  0.050783
33  1  0.868725  0.368725           0.414662  0.045937
26  1  0.863711  0.363711           0.318569  0.045142
34  1  1.320993  0.820993           0.778157  0.042836
22  1  0.8

In [39]:
treated_mean_Y = treated['Y'].mean()
matched_control_mean_Y = matched_controls['Y'].mean()
effect = treated_mean_Y - matched_control_mean_Y

print("treated_mean_Y:", treated_mean_Y)
print("(X=0):", matched_control_mean_Y)
print("Estimated Treatment Effect:", effect)

treated_mean_Y: 1.125597137873841
(X=0): 0.582237072655257
Estimated Treatment Effect: 0.5433600652185839


In [41]:
treated = df2[df2["X"] == 1].reset_index(drop=False)
control = df2[df2["X"] == 0].reset_index(drop=False)
nn = NearestNeighbors(radius=0.2)
nn.fit(control[["Z"]])
distances, indices = nn.radius_neighbors(treated[["Z"]])

matched_pairs = []

for treated_pos, control_positions in enumerate(indices):
    for j, control_pos in enumerate(control_positions):
        matched_pairs.append({
            "treated_original_index": treated.loc[treated_pos, "index"],
            "treated_Z": treated.loc[treated_pos, "Z"],
            "treated_Y": treated.loc[treated_pos, "Y"],
            "control_original_index": control.loc[control_pos, "index"],
            "control_Z": control.loc[control_pos, "Z"],
            "control_Y": control.loc[control_pos, "Y"],
            "distance": abs(treated.loc[treated_pos, "Z"] - control.loc[control_pos, "Z"])
        })

matched_pairs_df = pd.DataFrame(matched_pairs)

print(matched_pairs_df)

     treated_original_index  treated_Z  treated_Y  control_original_index  \
0                         1   0.715189   1.215189                       0   
1                         1   0.715189   1.215189                       2   
2                         1   0.715189   1.215189                       3   
3                         1   0.715189   1.215189                      11   
4                         1   0.715189   1.215189                      12   
..                      ...        ...        ...                     ...   
732                      98   0.828940   1.328940                      45   
733                      98   0.828940   1.328940                      56   
734                      98   0.828940   1.328940                      62   
735                      98   0.828940   1.328940                      74   
736                      98   0.828940   1.328940                      93   

     control_Z  control_Y  distance  
0     0.548814   0.548814  0.166376  

In [ ]:

matched_control_indices = []

for control_positions in indices:
    for control_pos in control_positions:
        matched_control_indices.append(
            control.loc[control_pos, "index"]
        )
counts = pd.Series(matched_control_indices).value_counts()
duplicates = (counts - 1).clip(lower=0).sum()

print("Duplicate matches:", duplicates)

Duplicate matches: 685


In [46]:

effects = []
for i, neighbor_indices in enumerate(indices):
    if len(neighbor_indices) == 0:
        continue
    treated_y = treated.loc[i, "Y"]
    control_mean_y = control.loc[neighbor_indices, "Y"].mean()
    effect = treated_y - control_mean_y
    effects.append(effect)

average_effect = np.mean(effects)

print("Individual effects:")
print(effects)

print("\nAverage treatment effect:")
print(average_effect)

Individual effects:
[np.float64(0.59880301930429), np.float64(0.5524101027227183), np.float64(0.493610678342714), np.float64(0.6471904892931742), np.float64(0.6855060095511788), np.float64(0.4742666405048309), np.float64(0.613915326148485), np.float64(0.666886473118585), np.float64(0.6353926977677643), np.float64(0.6439157972256), np.float64(0.621348852282544), np.float64(0.4822106240986367), np.float64(0.6027194643522761), np.float64(0.546437010983586), np.float64(0.6665121660997334), np.float64(0.6066102991378369), np.float64(0.5063033966387541), np.float64(0.5389824534387405), np.float64(0.5405727270699203), np.float64(0.6655913275647737), np.float64(0.5654339520353538), np.float64(0.48409737468894254), np.float64(0.5812448488591354), np.float64(0.4873506964872112), np.float64(0.5350923533565437), np.float64(0.4883002446315652), np.float64(0.5080662181879853), np.float64(0.49462498054234194), np.float64(0.5197339167173541), np.float64(0.4966073501809717), np.float64(0.55500676051241